Cell_01 Imports

In [1]:
import pandas as pd
from pathlib import Path
import sys

# add project root to path (so we can import config)
sys.path.insert(0, str(Path.cwd().parent))

# Now import from config after path is set up
from config import constants, settings

Cell 2: Load the CSV

In [2]:
# Cell 2: Load CSV
file_path = settings.INPUT_DIR / settings.INPUT_FILE
print(f"Loading from: {file_path}")

df= pd.read_csv(file_path)
print(f"loaded: {len(df)} rows, {len(df.columns)} columns")

Loading from: /home/alara/projects/lc_pipeline/data/input/lc_transactions_v3_shuffled.csv
loaded: 200 rows, 44 columns


Cell 3: Basic Shape & First Look

In [3]:
# Cell 3: First look at data
df.head(3)

,transaction_id,lc_number,lc_form,available_with,issue_date,expiry_date,latest_shipment_date,applicant_name,applicant_address,applicant_city,...,incoterm,port_of_loading,port_of_discharge,vessel_name,goods_description,documents_required,additional_conditions,presentation_period_days,payment_terms,charges
0,LC-ZCOTOH,LC-2026-115,STANDBY,BY MIXED PAYMENT,2026-01-11,2026-06-15,2026-05-24,Toyota Motor Corporation,1 Toyota-cho,Toyota City,...,FCA,DEHAM,USSFO,Vessel 15,Office Equipment and Supplies,Commercial Invoice; Packing List; Bill of Lading,NaN,14,AT SIGHT,OUR
1,LC-OH9SDB,LC—2026—148,STANDBY,BY ACCEPTANCE,2026-01-20,2026-05-23,2026-05-05,Apple Inc.,One Apple Park Way,Cupertino,...,FCA,NLRTM,USLAX,Vessel 48,Medical Equipment,Commercial Invoice; Packing List; Bill of Lading,NaN,21,60 DAYS AFTER SIGHT,SHA
2,LC-6GVWRD,LC—2026—138,STANDBY,BY DEFERRED PAYMENT,2026-01-16,2026-07-04,2026-06-14,Toyota Motor Credit,6565 Headquarters Dr,Plano,...,DAP,GBLGP,JPNGO,Vessel 38,Renewable Energy Components,Commercial Invoice; Packing List; Bill of Lading,NaN,14,60 DAYS AFTER SIGHT,SHA


Cell 4: Column names and data types

In [4]:
df.dtypes

transaction_id                str
lc_number                     str
lc_form                       str
available_with                str
issue_date                    str
expiry_date                   str
latest_shipment_date          str
applicant_name                str
applicant_address             str
applicant_city                str
applicant_postal_code         str
applicant_country             str
applicant_lei                 str
applicant_account             str
beneficiary_name              str
beneficiary_address           str
beneficiary_city              str
beneficiary_postal_code       str
beneficiary_country           str
beneficiary_lei               str
beneficiary_account           str
issuing_bank_name             str
issuing_bank_swift            str
issuing_bank_country          str
advising_bank_name            str
advising_bank_swift           str
confirming_bank_name          str
confirming_bank_swift         str
confirmation_status           str
amount        

Cell 5: Missing Values

In [5]:
# Cell 5: Missing values
missing = df.isnull().sum()
missing[missing > 0]

lc_number                  2
issue_date                 2
applicant_name             1
applicant_address          2
applicant_lei              1
beneficiary_lei            2
issuing_bank_swift         4
confirming_bank_name     148
confirming_bank_swift    148
port_of_loading           23
port_of_discharge          1
additional_conditions    175
dtype: int64

Cell 6: Explore Categorical Columns

In [6]:
# Cell 6a: Explore categorical columns
print("=== CURRENCY ===")
print(df['currency'].value_counts())

=== CURRENCY ===
currency
JPY    42
USD    39
EUR    38
GBP    32
jpy    16
gbp    15
usd    12
eur     6
Name: count, dtype: int64


In [7]:
# Cell 6b: More categorical columns

In [8]:
print("\n=== LC FORM ===")
print(df['lc_form'].value_counts())

print("\n=== INCOTERM ===")
print(df['incoterm'].value_counts())


=== LC FORM ===
lc_form
STANDBY                     75
IRREVOCABLE                 66
IRREVOCABLE TRANSFERABLE    58
INVALID_FORM                 1
Name: count, dtype: int64

=== INCOTERM ===
incoterm
CPT    24
FCA    22
DAP    20
DDP    19
CFR    19
EXW    18
DPU    18
FAS    17
CIF    16
FOB    15
CIP     9
XXX     3
Name: count, dtype: int64


Cell 6c: Confirmation status

In [9]:
print("\n=== CONFIRMATION STATUS ===")
print(df['confirmation_status'].value_counts())


=== CONFIRMATION STATUS ===
confirmation_status
UNCONFIRMED    175
CONFIRMED       24
MAYBE            1
Name: count, dtype: int64


In [10]:
# Cell 6d: Find the CONFIRMED transactions
df[df['confirmation_status'] == 'CONFIRMED'][['transaction_id', 'confirmation_status', 'confirming_bank_name', 'confirming_bank_swift']]

,transaction_id,confirmation_status,confirming_bank_name,confirming_bank_swift
0,LC-ZCOTOH,CONFIRMED,Barclays,BARCGB22
1,LC-OH9SDB,CONFIRMED,HSBC UK,HBUKGB4B
5,LC-L2W37D,CONFIRMED,HSBC UK,HBUKGB4B
37,LC-J2QP89,CONFIRMED,Sumitomo Mitsui,SMBCJPJT
74,LC-XO9T7E,CONFIRMED,MUFG Bank,BOTKJPJT
79,LC-AHXTHV,CONFIRMED,Shinhan Bank,SHBKKRSE
82,LC-DZR1YX,CONFIRMED,NaN,NaN
84,LC-DW5PY6,CONFIRMED,MUFG Bank,BOTKJPJT
98,LC-Y0BVR6,CONFIRMED,Deutsche Bank,DEUTDEFF
124,LC-3W5UZB,CONFIRMED,Sumitomo Mitsui,SMBCJPJT


Cell 7: Explore Amounts 💰

In [11]:
# Cell 7: Explore amounts
print("=== AMOUNT COLUMN ===")
print(f"Data type:  {df['amount'].dtype}")
print(df['amount'].head(10))


=== AMOUNT COLUMN ===
Data type:  str
0    49,543,534.00
1       43,942,455
2       701,960.00
3         12537943
4         31025296
5    44,698,977.00
6    29,573,500.00
7     6,721,792.00
8     4,994,878.00
9         37125991
Name: amount, dtype: str


 Cell 7b: Explore ALL amount patterns

In [12]:
# Cell 7b:
print("=== AMOUNT PATTERNS ===")
print(f"Total rows: {len(df)}")

# How many have commas? (thousand separators)
has_commas = df['amount'].str.contains(',', na=False).sum()
print(f"with commas: {has_commas}")

# How many have decimals?
has_decimals = df['amount'].str.contains(r'\.', na=False).sum()
print(f"With decimals: {has_decimals}")

# Sample of different formats
print("\n=== SAMPLE AMOUNTS ===")
print(df['amount'].head(15))


=== AMOUNT PATTERNS ===
Total rows: 200
with commas: 106
With decimals: 79

=== SAMPLE AMOUNTS ===
0     49,543,534.00
1        43,942,455
2        701,960.00
3          12537943
4          31025296
5     44,698,977.00
6     29,573,500.00
7      6,721,792.00
8      4,994,878.00
9          37125991
10         14155946
11    10,393,710.00
12    16,565,922.00
13          5457531
14    13,300,487.00
Name: amount, dtype: str


  Cell 8: Amount format patterns

In [13]:
# Pattern 1: Contains comma?
has_comma = df['amount'].str.contains(',', na=False)

# Pattern 2: Contains decimal point?
has_decimal = df['amount'].str.contains(r'\.', na=False)

# Pattern 3: Is it purely numeric (no formatting at all)?
pure_numeric = df['amount'].str.match(r'^\d+$', na=False)

# Pattern 4: Non-numeric garbage?
non_numeric = ~df['amount'].str.match(r'^[\d,.\-]+$', na=False)

print("=== AMOUNT FORMAT DISTRIBUTION ===")
print(f"With commas:      {has_comma.sum()}")
print(f"With decimals:    {has_decimal.sum()}")
print(f"Pure numeric:     {pure_numeric.sum()}")
print(f"Non-numeric:      {non_numeric.sum()}")

=== AMOUNT FORMAT DISTRIBUTION ===
With commas:      106
With decimals:    79
Pure numeric:     77
Non-numeric:      1


Cell 8b — find that non-numeric troublemaker

In [14]:
# Cell 8b: Find the non-numeric garbage
non_numeric_mask = ~df['amount'].str.match(r'^[\d,.\-]+$', na=False)
print("=== NON-NUMERIC AMOUNTS ===")
print(df[non_numeric_mask][['transaction_id', 'amount']])


=== NON-NUMERIC AMOUNTS ===
   transaction_id   amount
47      LC-9KWV08  INVALID


Cell 8c: Inspect comma patterns

In [15]:
# Cell 8c: Inspect comma patterns
has_comma = df.loc[df['amount'].str.contains(',', na = False), 'amount']
print(f"=== AMOUNTS WITH COMMAS ({len(has_comma)}) ===")
print("\nSample of 15:")
print(has_comma.sample(15, random_state=42).tolist())

=== AMOUNTS WITH COMMAS (106) ===

Sample of 15:
['30,288,932.00', '33,048,894.00', '29,573,500.00', '19,313,955.00', '16,183,068.00', '14,296,389', '29,236,861.00', '43,931,390.00', '46,633,126.00', '24,349,872.00', '32,667,655', '12,846,310.00', '49,543,534.00', '9,015,497', '43,816,936.00']


 Cell 8d — find suspicious comma patterns

In [16]:
# Cell 8d: Hunt for malformed comma patterns
amounts_with_comma = df[df['amount'].str.contains(',', na=False)]


# Good pattern: commas every 3 digits from the right
# Bad patterns: anything else!

good_comma_pattern = r'^\d{1,3}(,\d{3})*(\.\d+)?$'
suspect = amounts_with_comma[
    ~amounts_with_comma['amount'].str.match(good_comma_pattern, na=False)
]

print(f"=== MALFORMED COMMA PATTERNS ({len(suspect)}) ===")
print(suspect[['transaction_id', 'amount']])


=== MALFORMED COMMA PATTERNS (6) ===
    transaction_id       amount
86       LC-94874F    22,43,565
111      LC-36KOF8     123,,456
112      LC-L6246H     123,456,
132      LC-4QXXVA  1,00,00,000
133      LC-JLTEIY   12,3456,78
174      LC-1UDBFW     ,123,456


Cell 8e — hunt for more amount problems

In [17]:
# Cell 8e: Other amount edge cases
print("=== OTHER AMOUNT EDGE CASES ===\n")

# Zero amounts
zeros = df.loc[df['amount'] == '0', ['transaction_id', 'amount']]
print(f"Zero amounts ({len(zeros)}):")
print(zeros)

# Negative amounts (starts with -)
negatives = df.loc[df['amount'].str.startswith('-', na=False), ['transaction_id', 'amount']]
print(f"\nNegative amounts ({len(negatives)}):")
print(negatives)

# Too many decimal places (more than 2)
too_many_decimals = df.loc[
    df['amount'].str.contains(r'\.\d{3,}', na=False),
    ['transaction_id', 'amount']
]
print(f"\nToo many decimals ({len(too_many_decimals)}):")
print(too_many_decimals)

=== OTHER AMOUNT EDGE CASES ===

Zero amounts (3):
    transaction_id amount
56       LC-PJA1WJ      0
97       LC-NAM148      0
183      LC-GATJ9T      0

Negative amounts (11):
    transaction_id  amount
26       LC-D14VE9  -50000
44       LC-JZA7TZ   -1000
71       LC-D84U89   -1000
83       LC-UUZPEA   -1000
90       LC-HE7UR2   -1000
96       LC-ZE5R5U   -1000
103      LC-ZP08J3   -1000
105      LC-9SKDGI  -50000
116      LC-3WXA73   -1000
175      LC-0TPAC5   -1000
191      LC-3RZSJ4  -50000

Too many decimals (3):
    transaction_id     amount
27       LC-JIUJV6      0.001
29       LC-HAIR4C  12345.123
128      LC-840982      0.001


Cell 8f: Extremely large amounts

In [18]:
# Cell 8f: Extremely large amounts
# First, let's see the biggest values (clean ones we can parse)

clean_amounts = df.loc[
    df['amount'].str.match(r'^[\d,]+(\.\d+)?$', na=False),
    'amount'
]

# Remove commas and convert to float for comparison
numeric_vals = clean_amounts.str.replace(',', '').astype(float)

print("=== LARGEST AMOUNTS ===")
print(numeric_vals.nlargest(5))

print("\n=== CHECK: Any exceed 999,999,999.99? ===")
exceeds_max = numeric_vals[numeric_vals > 999_999_999.99]
print(f"Count: {len(exceeds_max)}")
if len(exceeds_max) > 0:
    print(exceeds_max)


=== LARGEST AMOUNTS ===
73     1.000000e+10
107    1.000000e+10
43     4.998762e+07
177    4.996333e+07
144    4.975824e+07
Name: amount, dtype: float64

=== CHECK: Any exceed 999,999,999.99? ===
Count: 2
73     1.000000e+10
107    1.000000e+10
Name: amount, dtype: float64


Cell 8g: Find the oversized amounts

In [19]:
# Cell 8g: Find the oversized amounts
large_mask = df['amount'].str.replace(',', '', regex=False).str.match(r'^\d+\.?\d*$', na=False)
#  regex=False is the explicit, correct way to say "this is just a character, not a pattern"
large_df = df.loc[large_mask].copy()
large_df['numeric'] = large_df['amount'].str.replace(',', '').astype(float)

oversized = large_df.loc[large_df['numeric'] > 999_999_999.99, ['transaction_id', 'amount']]
print("=== AMOUNTS EXCEEDING MAXIMUM ===")
print(oversized)


=== AMOUNTS EXCEEDING MAXIMUM ===
    transaction_id         amount
73       LC-KRD5EQ  9999999999.99
107      LC-GNZS43  9999999999.99


# LEI CODES

Cell 9: LEI exploration - basic stats

In [20]:
# Cell 9: LEI exploration - basic stats
print("=== LEI COLUMNS ===\n")

print("APPLICANT LEI:")
print(f"  Missing: {df['applicant_lei'].isna().sum()}")
print(f"  Present: {df['applicant_lei'].notna().sum()}")

print("\nBENEFICIARY LEI:")
print(f"  Missing: {df['beneficiary_lei'].isna().sum()}")
print(f"  Present: {df['beneficiary_lei'].notna().sum()}")

# Check lengths
print("\n=== LEI LENGTHS ===")
print("\nApplicant LEI lengths:")
print(df['applicant_lei'].str.len().value_counts().sort_index())

=== LEI COLUMNS ===

APPLICANT LEI:
  Missing: 1
  Present: 199

BENEFICIARY LEI:
  Missing: 2
  Present: 198

=== LEI LENGTHS ===

Applicant LEI lengths:
applicant_lei
11.0      2
20.0    197
Name: count, dtype: int64


Cell 9b: Find LEI length problems

In [21]:
print("=== APPLICANT LEI - WRONG LENGTH ===")
#  the index is always returned automatically
wrong_len_app = df.loc[ # call filters rows and selects specific columns
    df['applicant_lei'].str.len() != 20, # row filter
    ['transaction_id', 'applicant_lei'] # column selector
]
print(wrong_len_app)

print("\n=== BENEFICIARY LEI LENGTHS ===")
print(df['beneficiary_lei'].str.len().value_counts().sort_index())

print("\n=== BENEFICIARY LEI - WRONG LENGTH ===")
wrong_len_ben = df.loc[
    df['beneficiary_lei'].str.len() != 20,
    ['transaction_id', 'beneficiary_lei']
]
print(wrong_len_ben)

=== APPLICANT LEI - WRONG LENGTH ===
    transaction_id applicant_lei
50       LC-TRP0J4   TOOSHORT123
148      LC-Q85JSG   TOOSHORT123
162      LC-L4ZKLO           NaN

=== BENEFICIARY LEI LENGTHS ===
beneficiary_lei
20.0    198
Name: count, dtype: int64

=== BENEFICIARY LEI - WRONG LENGTH ===
    transaction_id beneficiary_lei
93       LC-6WKTAK             NaN
150      LC-URTP0L             NaN


Cell 9c — validate format of the 20-character LEIs:

In [22]:
# Cell 9c: Check LEI FORMAT (not just length)
import re
from config.constants import PATTERNS

lei_pattern = PATTERNS['Lei']

# Check applicant LEIs that have 20 chars
valid_len_app = df.loc[df['applicant_lei'].str.len() == 20, 'applicant_lei' ]

# How many match the pattern?
matches_pattern = valid_len_app.str.match(lei_pattern).sum()
fails_pattern = len(valid_len_app) - matches_pattern

print("=== APPLICANT LEI FORMAT CHECK (20-char only) ===")
print(f"Matches pattern: {matches_pattern}")
print(f"Fails pattern:   {fails_pattern}")

# Show the failures
if fails_pattern > 0:
    print("\n=== INVALID FORMAT ===")
    bad_format = df.loc[
        (df['applicant_lei'].str.len() == 20) &
        (~df['applicant_lei'].str.match(lei_pattern)),
        ['transaction_id', 'applicant_lei']
    ]
    print(bad_format)

=== APPLICANT LEI FORMAT CHECK (20-char only) ===
Matches pattern: 196
Fails pattern:   1

=== INVALID FORMAT ===
    transaction_id         applicant_lei
182      LC-3CFL0U  INVALID!!LEI##CODE99


 **Cell 9d: Check BENEFICIARY LEI format**

In [23]:
# Cell 9d: Check BENEFICIARY LEI format
valid_len_ben = df.loc[df['beneficiary_lei'].str.len() == 20, 'beneficiary_lei']

matches_pattern = valid_len_ben.str.match(lei_pattern).sum()
fails_pattern = len(valid_len_ben) - matches_pattern

print("=== BENEFICIARY LEI FORMAT CHECK (20-char only) ===")
print(f"Matches pattern: {matches_pattern}")
print(f"Fails pattern:   {fails_pattern}")

if fails_pattern > 0:
    print("\n=== INVALID FORMAT ===")
    bad_format = df.loc[
        (df['beneficiary_lei'].str.len() == 20) &
        (~df['beneficiary_lei'].str.match(lei_pattern)),
        ['transaction_id', 'beneficiary_lei']
    ]
    print(bad_format)

=== BENEFICIARY LEI FORMAT CHECK (20-char only) ===
Matches pattern: 198
Fails pattern:   0


# 10 **SWIFT/BIC Codes**

Cell 10: SWIFT/BIC exploration - basic stats

In [24]:
print("=== SWIFT/BIC COLUMNS ===\n")
swift_cols = ['issuing_bank_swift', 'advising_bank_swift', 'confirming_bank_swift']

for col in swift_cols:
    missing = df[col].isna().sum()
    present = df[col].notna().sum()
    print(f"{col}:")
    print(f"  Missing: {missing}")
    print(f"  Present: {present}\n")

=== SWIFT/BIC COLUMNS ===

issuing_bank_swift:
  Missing: 4
  Present: 196

advising_bank_swift:
  Missing: 0
  Present: 200

confirming_bank_swift:
  Missing: 148
  Present: 52



Cell 10b

In [25]:
# Cell 10b: SWIFT length check
print("=== ISSUING BANK SWIFT - LENGTHS ===")
print(df['issuing_bank_swift'].str.len().value_counts().sort_index()) # sort_index() -> descending

print("\n=== ADVISING BANK SWIFT - LENGTHS ===")
print(df['advising_bank_swift'].str.len().value_counts().sort_index())

print("\n=== CONFIRMING BANK SWIFT - LENGTHS ===")
print(df['confirming_bank_swift'].str.len().value_counts().sort_index())

# Critical check: CONFIRMED but no confirming bank?
print("\n=== CONFIRMED WITHOUT CONFIRMING BANK ===")
confirmed_no_bank = df.loc[
    (df['confirmation_status'] == 'CONFIRMED') &
    (df['confirming_bank_swift'].isna()),
    ['transaction_id', 'confirmation_status', 'confirming_bank_swift']
]
print(confirmed_no_bank)

=== ISSUING BANK SWIFT - LENGTHS ===
issuing_bank_swift
5.0      3
8.0    193
Name: count, dtype: int64

=== ADVISING BANK SWIFT - LENGTHS ===
advising_bank_swift
8    200
Name: count, dtype: int64

=== CONFIRMING BANK SWIFT - LENGTHS ===
confirming_bank_swift
8.0    52
Name: count, dtype: int64

=== CONFIRMED WITHOUT CONFIRMING BANK ===
    transaction_id confirmation_status confirming_bank_swift
82       LC-DZR1YX           CONFIRMED                   NaN
137      LC-A26YFP           CONFIRMED                   NaN


Cell 10c-extra: Confirmation cross-validation

In [26]:
# Cell 10c-extra: Confirmation cross-validation
print("=== CONFIRMATION STATUS VALUES ===")
print(df['confirmation_status'].value_counts())

print("\n=== SCENARIO 1: CONFIRMED but NO bank ===")
confirmed_no_bank = df.loc[
    (df['confirmation_status'] == 'CONFIRMED') &
    (df['confirming_bank_swift'].isna()),
    ['transaction_id', 'confirmation_status', 'confirming_bank_swift']
]
print(f"Count: {len(confirmed_no_bank)}")
print(confirmed_no_bank)

print("\n=== SCENARIO 2: Has bank but NOT confirmed ===")
bank_not_confirmed = df.loc[
    (df['confirming_bank_swift'].notna()) &
    (df['confirmation_status'] != 'CONFIRMED'),
    ['transaction_id', 'confirmation_status', 'confirming_bank_swift']
]
print(f"Count: {len(bank_not_confirmed)}")
print(bank_not_confirmed)

print("\n=== SCENARIO 3: Invalid confirmation status ===")
valid_statuses = ['CONFIRMED', 'UNCONFIRMED']
invalid_status = df.loc[
    ~df['confirmation_status'].isin(valid_statuses),
    ['transaction_id', 'confirmation_status']
]
print(f"Count: {len(invalid_status)}")
print(invalid_status)

=== CONFIRMATION STATUS VALUES ===
confirmation_status
UNCONFIRMED    175
CONFIRMED       24
MAYBE            1
Name: count, dtype: int64

=== SCENARIO 1: CONFIRMED but NO bank ===
Count: 2
    transaction_id confirmation_status confirming_bank_swift
82       LC-DZR1YX           CONFIRMED                   NaN
137      LC-A26YFP           CONFIRMED                   NaN

=== SCENARIO 2: Has bank but NOT confirmed ===
Count: 30
    transaction_id confirmation_status confirming_bank_swift
14       LC-T1GHR0         UNCONFIRMED              SHBKKRSE
16       LC-JGZLMA         UNCONFIRMED              SHBKKRSE
18       LC-1HT7PZ         UNCONFIRMED              BOTKJPJT
19       LC-8OZCYW         UNCONFIRMED              CITIUS33
25       LC-1ERTJ5         UNCONFIRMED              BOTKJPJT
51       LC-6T4UFE         UNCONFIRMED              CHASUS33
58       LC-L8OS9X         UNCONFIRMED              CITIUS33
64       LC-PBMYOF         UNCONFIRMED              BARCGB22
68       LC-YZX4W6  

In [27]:
# Cell 10c-blank: Check for blank confirmation status
print("=== BLANK CONFIRMATION STATUS ===")
blank_status = df.loc[
    df['confirmation_status'].isna() |
    (df['confirmation_status'].str.strip() == ''),
    ['transaction_id', 'confirmation_status', 'confirming_bank_swift']
]
print(f"Count: {len(blank_status)}")
print(blank_status)

=== BLANK CONFIRMATION STATUS ===
Count: 0
Empty DataFrame
Columns: [transaction_id, confirmation_status, confirming_bank_swift]
Index: []


Cell 10c: Find short/invalid SWIFT codes

In [28]:
# Cell 10c: Find short/invalid SWIFT codes (no hardcoded values!)
from config.constants import PATTERNS
from config.validation_rules import SWIFT_RULES

swift_pattern = PATTERNS['swift_bic']
valid_lengths = SWIFT_RULES['valid_lengths']

print(f"Valid SWIFT lengths: {valid_lengths}")

print("\n=== ISSUING SWIFT - WRONG LENGTH ===")
wrong_len = df.loc[
    ~df['issuing_bank_swift'].str.len().isin(valid_lengths) |
    df['issuing_bank_swift'].isna(),
    ['transaction_id', 'issuing_bank_swift']
]
print(wrong_len)

print("\n=== ISSUING SWIFT - FORMAT CHECK ===")
valid_len = df.loc[df['issuing_bank_swift'].str.len().isin(valid_lengths), 'issuing_bank_swift']
matches = valid_len.str.match(swift_pattern).sum()
fails = len(valid_len) - matches

print(f"Matches pattern: {matches}")
print(f"Fails pattern:   {fails}")

if fails > 0:
    print("\n=== INVALID FORMAT ===")
    bad_format = df.loc[
        (df['issuing_bank_swift'].str.len().isin(valid_lengths)) &
        (~df['issuing_bank_swift'].str.match(swift_pattern)),
        ['transaction_id', 'issuing_bank_swift']
    ]
    print(bad_format)

Valid SWIFT lengths: [8, 11]

=== ISSUING SWIFT - WRONG LENGTH ===
    transaction_id issuing_bank_swift
60       LC-UK9E1V                NaN
66       LC-1B0Z3N              SHORT
69       LC-C5BA75                NaN
91       LC-OWGZRI                NaN
110      LC-9K9XJU                NaN
172      LC-N1U349              SHORT
176      LC-30T9NT              SHORT

=== ISSUING SWIFT - FORMAT CHECK ===
Matches pattern: 191
Fails pattern:   2

=== INVALID FORMAT ===
    transaction_id issuing_bank_swift
117      LC-PHT0HL           12345678
120      LC-QEWAOU           12345678


# 11  DATES

Cell 11: Date exploration - basic stats

In [29]:
# Cell 11: Date exploration - basic stats
print("=== DATE COLUMNS ===\n")

date_cols = ['issue_date', 'expiry_date', 'latest_shipment_date']

for col in date_cols:
    missing = df[col].isna().sum()
    present = df[col].notna().sum()
    print(f"{col}:")
    print(f"  Missing: {missing}")
    print(f"  Present: {present}\n")

=== DATE COLUMNS ===

issue_date:
  Missing: 2
  Present: 198

expiry_date:
  Missing: 0
  Present: 200

latest_shipment_date:
  Missing: 0
  Present: 200



Cell 11b: Missing dates + date sequence validation

In [30]:
# Cell 11b: Missing dates + date sequence validation
print("=== MISSING ISSUE DATES ===")
missing_issue = df.loc[
    df['issue_date'].isna(),
    ['transaction_id', 'issue_date', 'expiry_date', 'latest_shipment_date']
]
print(missing_issue)

# Convert to datetime for comparison (only non-null rows)
# errors='coerce' turns impossible dates (e.g. 2026-02-30) into NaT instead of crashing
print("\n=== DATE SEQUENCE CHECK ===")
df_dates = df.loc[df['issue_date'].notna()].copy()
df_dates['issue_dt']    = pd.to_datetime(df_dates['issue_date'],           format='mixed', errors='coerce')
df_dates['expiry_dt']   = pd.to_datetime(df_dates['expiry_date'],          format='mixed', errors='coerce')
df_dates['shipment_dt'] = pd.to_datetime(df_dates['latest_shipment_date'], format='mixed', errors='coerce')

# Report any dates that couldn't be parsed (impossible/garbage dates)
for col, dt_col in [('issue_date', 'issue_dt'), ('expiry_date', 'expiry_dt'), ('latest_shipment_date', 'shipment_dt')]:
    bad = df_dates.loc[df_dates[dt_col].isna(), ['transaction_id', col]]
    if len(bad):
        print(f"\nINVALID dates in {col} ({len(bad)}):")
        print(bad)

# DATE004: Expiry before issue?
expiry_before_issue = df_dates.loc[
    df_dates['expiry_dt'] < df_dates['issue_dt'],
    ['transaction_id', 'issue_date', 'expiry_date']
]
print(f"\nExpiry BEFORE issue (DATE004): {len(expiry_before_issue)}")
print(expiry_before_issue)

# DATE005: Shipment after expiry?
shipment_after_expiry = df_dates.loc[
    df_dates['shipment_dt'] > df_dates['expiry_dt'],
    ['transaction_id', 'expiry_date', 'latest_shipment_date']
]
print(f"\nShipment AFTER expiry (DATE005): {len(shipment_after_expiry)}")
print(shipment_after_expiry)

=== MISSING ISSUE DATES ===
    transaction_id issue_date expiry_date latest_shipment_date
121      LC-30ME8F        NaN  2026-05-26           2026-05-05
146      LC-B2KNFT        NaN  2026-05-23           2026-04-26

=== DATE SEQUENCE CHECK ===

INVALID dates in issue_date (1):
   transaction_id  issue_date
94      LC-WM31YI  2026-02-30

Expiry BEFORE issue (DATE004): 2
    transaction_id  issue_date expiry_date
106      LC-PUXQPH  2026-06-01  2026-03-01
187      LC-PXNF7C  2026-06-01  2026-03-01

Shipment AFTER expiry (DATE005): 7
    transaction_id expiry_date latest_shipment_date
28       LC-P0TVHH  2026-06-15           2026-08-20
42       LC-ODRO8B  2026-06-01           2026-08-01
76       LC-65KXVF  2026-06-01           2026-08-01
87       LC-CW790P  2026-06-15           2026-08-20
106      LC-PUXQPH  2026-03-01           2026-05-11
156      LC-38DALY  2026-06-15           2026-08-20
187      LC-PXNF7C  2026-03-01           2026-05-08


Cell 11c: LC validity period check (max 540 days)

In [31]:
# Cell 11c: LC validity period check (max 540 days)
from config.validation_rules import DATE_RULES

max_validity = DATE_RULES['max_validity_days']
print(f"Max validity period: {max_validity} days\n")

# Calculate validity period (expiry - issue)
df_dates['validity_days'] = (df_dates['expiry_dt'] - df_dates['issue_dt']).dt.days

# Find excessive validity periods
excessive = df_dates.loc[ # .loc[row_filter, column_list]
    df_dates['validity_days'] > max_validity,
    ['transaction_id', 'issue_date', 'expiry_date', 'validity_days']
]
print(f"=== VALIDITY EXCEEDS {max_validity} DAYS (DATE007) ===")
print(f"Count: {len(excessive)}")
print(excessive)

# Also show normal range for context
print(f"\n=== VALIDITY PERIOD STATS ===")
print(df_dates['validity_days'].describe())


Max validity period: 540 days

=== VALIDITY EXCEEDS 540 DAYS (DATE007) ===
Count: 4
    transaction_id  issue_date expiry_date  validity_days
3        LC-0OXFQ8  2026-01-10  2028-06-10          882.0
15       LC-6DPBHS  2019-01-15  2026-05-28         2690.0
67       LC-5IGQPK  2026-01-10  2028-06-10          882.0
175      LC-0TPAC5  2026-01-10  2028-06-10          882.0

=== VALIDITY PERIOD STATS ===
count     197.000000
mean      171.908629
std       203.929314
min       -92.000000
25%       135.000000
50%       154.000000
75%       165.000000
max      2690.000000
Name: validity_days, dtype: float64


# PARTY

Cell 12: Party fields - critical checks

In [32]:
# Cell 12: Party fields - critical checks
print("=== CRITICAL PARTY FIELDS ===\n")

critical_fields = [
    'applicant_name', 'applicant_country',
    'beneficiary_name', 'beneficiary_country'
]
# reports two distinct quality checks per column: Missing (NaN) and Empty string.
for col in critical_fields:
    missing = df[col].isna().sum()
    empty = (df[col].str.strip() == '').sum() if df[col].dtype == 'object' else 0
    print(f"{col}:")
    print(f"  Missing (NaN): {missing}")
    print(f"  Empty string:  {empty}\n")

print("=== OTHER PARTY FIELDS ===\n")

other_fields = [
    'applicant_address', 'applicant_city', 'applicant_postal_code',
    'beneficiary_address', 'beneficiary_city', 'beneficiary_postal_code'
]

for col in other_fields:
    missing = df[col].isna().sum()
    print(f"{col}: {missing} missing")

=== CRITICAL PARTY FIELDS ===

applicant_name:
  Missing (NaN): 1
  Empty string:  0

applicant_country:
  Missing (NaN): 0
  Empty string:  0

beneficiary_name:
  Missing (NaN): 0
  Empty string:  0

beneficiary_country:
  Missing (NaN): 0
  Empty string:  0

=== OTHER PARTY FIELDS ===

applicant_address: 2 missing
applicant_city: 0 missing
applicant_postal_code: 0 missing
beneficiary_address: 0 missing
beneficiary_city: 0 missing
beneficiary_postal_code: 0 missing


Cell 12b: Country code validation

In [33]:
# Cell 12b: Country code validation
from config.constants import VALID_COUNTRY_CODES

print(f"Valid country codes: {len(VALID_COUNTRY_CODES)} countries")
print(f"Examples: {list(VALID_COUNTRY_CODES)[:10]}\n")

print("=== APPLICANT COUNTRY VALUES ===")
print(df['applicant_country'].value_counts())

print("\n=== INVALID COUNTRY CODES ===")
invalid_countries = df.loc[
    ~df['applicant_country'].isin(VALID_COUNTRY_CODES),
    ['transaction_id', 'applicant_country']
]
print(f"Count: {len(invalid_countries)}")
print(invalid_countries)

Valid country codes: 42 countries
Examples: ['BR', 'AE', 'KR', 'CO', 'TW', 'PE', 'FI', 'TH', 'KE', 'NO']

=== APPLICANT COUNTRY VALUES ===
applicant_country
US    81
KR    28
DE    26
JP    22
CH    16
GB    15
TW    11
XX     1
Name: count, dtype: int64

=== INVALID COUNTRY CODES ===
Count: 1
   transaction_id applicant_country
33      LC-XVG0FN                XX


Cell 12c: Beneficiary country validation

In [34]:
# Cell 12c: Beneficiary country validation
print("=== BENEFICIARY COUNTRY VALUES ===")
print(df['beneficiary_country'].value_counts())

print("\n=== INVALID BENEFICIARY COUNTRY CODES ===")
invalid_ben_countries = df.loc[
    ~df['beneficiary_country'].isin(VALID_COUNTRY_CODES),
    ['transaction_id', 'beneficiary_country']
]
print(f"Count: {len(invalid_ben_countries)}")
print(invalid_ben_countries)

=== BENEFICIARY COUNTRY VALUES ===
beneficiary_country
US    68
GB    32
DE    27
KR    22
TW    21
JP    20
CH    10
Name: count, dtype: int64

=== INVALID BENEFICIARY COUNTRY CODES ===
Count: 0
Empty DataFrame
Columns: [transaction_id, beneficiary_country]
Index: []


Cell 12d: Party name validation

In [35]:
# Cell 12d: Party name validation
from config.validation_rules import PARTY_RULES

min_len = PARTY_RULES['min_name_length']
max_len = PARTY_RULES['max_name_length']

print(f"Name length rules: {min_len} - {max_len} chars\n")

# Find the missing applicant name first
print("=== MISSING APPLICANT NAME (PTY001) ===")
missing_name = df.loc[
    df['applicant_name'].isna(),
    ['transaction_id', 'applicant_name']
]
print(missing_name)

# Check name lengths (exclude NaN)
print("\n=== APPLICANT NAME LENGTH CHECK ===")
app_names = df.loc[df['applicant_name'].notna(), 'applicant_name']
app_lens = app_names.str.len()

too_short = df.loc[
    (df['applicant_name'].notna()) &
    (df['applicant_name'].str.len() < min_len),
    ['transaction_id', 'applicant_name']
]
print(f"Too short (< {min_len}): {len(too_short)}")
if len(too_short) > 0:
    print(too_short)

too_long = df.loc[
    (df['applicant_name'].notna()) &
    (df['applicant_name'].str.len() > max_len),
    ['transaction_id', 'applicant_name']
]
print(f"Too long (> {max_len}): {len(too_long)}")
if len(too_long) > 0:
    print(too_long)

print(f"\n=== NAME LENGTH STATS ===")
print(app_lens.describe())

Name length rules: 2 - 140 chars

=== MISSING APPLICANT NAME (PTY001) ===
    transaction_id applicant_name
139      LC-HHXFGC            NaN

=== APPLICANT NAME LENGTH CHECK ===
Too short (< 2): 3
    transaction_id applicant_name
114      LC-X2X18H              A
165      LC-AXEQBN              A
181      LC-T63J3R              A
Too long (> 140): 0

=== NAME LENGTH STATS ===
count    199.000000
mean      16.040201
std        5.924473
min        1.000000
25%       11.000000
50%       16.000000
75%       20.000000
max       38.000000
Name: applicant_name, dtype: float64


Cell 12e: Beneficiary name validation

In [36]:
# Cell 12e: Beneficiary name validation
print("=== MISSING BENEFICIARY NAME (PTY001) ===")
missing_ben = df.loc[
    df['beneficiary_name'].isna(),
    ['transaction_id', 'beneficiary_name']
]
print(f"Count: {len(missing_ben)}")
print(missing_ben)

print("\n=== BENEFICIARY NAME LENGTH CHECK ===")
too_short_ben = df.loc[
    (df['beneficiary_name'].notna()) &
    (df['beneficiary_name'].str.len() < min_len),
    ['transaction_id', 'beneficiary_name']
]
print(f"Too short (< {min_len}): {len(too_short_ben)}")
if len(too_short_ben) > 0:
    print(too_short_ben)

too_long_ben = df.loc[
    (df['beneficiary_name'].notna()) &
    (df['beneficiary_name'].str.len() > max_len),
    ['transaction_id', 'beneficiary_name']
]
print(f"Too long (> {max_len}): {len(too_long_ben)}")
if len(too_long_ben) > 0:
    print(too_long_ben)

=== MISSING BENEFICIARY NAME (PTY001) ===
Count: 0
Empty DataFrame
Columns: [transaction_id, beneficiary_name]
Index: []

=== BENEFICIARY NAME LENGTH CHECK ===
Too short (< 2): 0
Too long (> 140): 0
